# C3 – Day 2: Preprocessing & Participant-wise Split
**AnchorAI | Component 3: Cognitive Checkpointing**

In Day 1 we explored the data. Today we:
1. Remove outliers (extreme RT values)
2. Select clean features
3. Encode categorical columns
4. Split data by participant (not randomly)
5. Save the preprocessed data to Drive

> Run on **CPU runtime** in Google Colab.

## Step 0 – Mount Drive and reload data
We reload the raw data exactly as in Day 1.

In [ ]:
import os, csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/C3/data/raw/hirsch'
    OUT_DIR  = '/content/drive/MyDrive/C3/data/processed'
    print('Colab – Drive mounted.')
except ImportError:
    DATA_DIR = r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing\data\raw\hirsch'
    OUT_DIR  = r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing\data\processed'
    print('Running locally.')

os.makedirs(OUT_DIR, exist_ok=True)

def load_hirsch_csv(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_lines = f.readlines()
    def parse_line(line):
        line = line.strip()
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1].replace('""', '"')
        return next(csv.reader([line]))
    header = parse_line(raw_lines[0])
    rows   = [parse_line(l) for l in raw_lines[1:] if l.strip()]
    return pd.DataFrame(rows, columns=header)

frames = []
for exp_name, fname in [('Exp1','raw-data_Exp1.csv'),('Exp2','raw-data_Exp2.csv'),('Exp3','raw-data_Exp3.csv')]:
    tmp = load_hirsch_csv(os.path.join(DATA_DIR, fname))
    tmp['experiment'] = exp_name
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
print(f'Raw data loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')

## Step 1 – Merge column-name variants and convert numeric columns
All columns loaded as strings. Two things to fix:
1. **Name mismatch:** Exp2 calls the column `Trialtype` (capital T), Exp1/Exp3 call it `trialtype`. If we do not merge them, every Exp2 row looks "missing" and gets dropped later (we lost Exp2 this way the first time). We merge them into one column.
2. Convert the numeric columns from text to numbers.

We also print rows and participants per experiment as a sanity check: all three experiments must appear.

In [ ]:
# Exp2 names this column 'Trialtype' (capital T); Exp1/Exp3 use 'trialtype'.
# After concat they became two separate columns, so merge them into one.
df['trialtype'] = df['trialtype'].fillna(df['Trialtype'])

numeric_cols = ['pp', 'blocktype', 'trial', 'trialtype', 'task', 'sequence', 'RT', 'error']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print('Rows before cleaning:', len(df))
print('RT missing:', df['RT'].isna().sum())
print('trialtype missing after merge (must be 0):', df['trialtype'].isna().sum())
print()
print('Rows per experiment:')
print(df['experiment'].value_counts())
print()
print('Participants per experiment:')
print(df.groupby('experiment')['pp'].nunique())

## Step 2 – Remove outliers

**Why?** In Day 1 we saw RT max = 417,961 ms (~7 minutes). A real task resumption takes seconds, not minutes. Extreme values are likely measurement artifacts (participant walked away, computer froze, etc.).

**Method:** We use the IQR (Interquartile Range) rule:
- Q1 = 25th percentile, Q3 = 75th percentile
- IQR = Q3 – Q1
- Outlier threshold = Q3 + 3 × IQR (we use 3× because RT is naturally right-skewed)

We also remove RT < 100ms (too fast to be a real decision — likely a keypress error).

In [ ]:
rt_clean = df['RT'].dropna()
Q1  = rt_clean.quantile(0.25)
Q3  = rt_clean.quantile(0.75)
IQR = Q3 - Q1
upper_limit = Q3 + 3 * IQR
lower_limit = 100  # ms minimum

print(f'Q1={Q1:.0f} ms,  Q3={Q3:.0f} ms,  IQR={IQR:.0f} ms')
print(f'Outlier upper limit : {upper_limit:.0f} ms')
print(f'Outlier lower limit : {lower_limit} ms')

before = len(df)
df = df[df['RT'].notna() & (df['RT'] >= lower_limit) & (df['RT'] <= upper_limit)]
after = len(df)

print(f'\nRows removed : {before - after:,}  ({(before-after)/before*100:.1f}%)')
print(f'Rows remaining: {after:,}')

# Show cleaned RT distribution
plt.figure(figsize=(8,4))
plt.hist(df['RT'], bins=60, color='steelblue', edgecolor='white')
plt.title('RT Distribution after Outlier Removal')
plt.xlabel('Reaction Time (ms)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('rt_after_outlier_removal.png', dpi=150)
plt.show()
print(f'Cleaned RT range: {df["RT"].min():.0f} – {df["RT"].max():.0f} ms')

## Step 3 – Re-create Low / Medium / High labels

**Important:** We recalculate terciles on the CLEANED data (after outlier removal). If we used the old terciles from Day 1, the boundaries would be skewed by the extreme values.

In [ ]:
t33 = df['RT'].quantile(0.333)
t67 = df['RT'].quantile(0.667)

def assign_label(rt_val):
    if rt_val <= t33:  return 'Low'
    elif rt_val <= t67: return 'Medium'
    else:               return 'High'

df['difficulty'] = df['RT'].apply(assign_label)

print(f'New tercile boundaries:')
print(f'  Low    : RT <= {t33:.0f} ms')
print(f'  Medium : {t33:.0f} < RT <= {t67:.0f} ms')
print(f'  High   : RT > {t67:.0f} ms')
print()
print('Label distribution:')
print(df['difficulty'].value_counts())

## Step 4 – Select features

**Why these features?**
- `blocktype` (1/2): whether this was an interruption block — directly related to difficulty
- `task` (0–4): different task types have different cognitive loads
- `trialtype` (1/2): type of trial within a block
- `sequence` (1/2): order of task — fatigue and learning effects
- `trial`: trial number — fatigue over time
- `error` (0/1): whether participant made an error — signal of cognitive load
- `experiment` (Exp1/2/3): experiment ID — controls for between-experiment differences

**Dropped columns:**
- `pp` — participant ID, not a feature (data leakage)
- `RT` — target variable itself
- `bBlock`, `Stimulus1/2/3/4`, `Color1/2/3/4`, `Response`, `response`, `Errortyp` — too many missing / not relevant

In [ ]:
FEATURE_COLS = ['blocktype', 'task', 'trialtype', 'sequence', 'trial', 'error', 'experiment']
TARGET_COL   = 'difficulty'
GROUP_COL    = 'pp'

# Keep only what we need
df_model = df[FEATURE_COLS + [TARGET_COL, GROUP_COL]].copy()

# Drop rows with any missing feature value
before = len(df_model)
df_model = df_model.dropna(subset=FEATURE_COLS + [TARGET_COL])
print(f'Rows dropped due to missing features: {before - len(df_model):,}')
print(f'Rows remaining for modelling: {len(df_model):,}')
print()
df_model.head()

## Step 5 – Encode categorical columns

Machine learning models need **numbers**, not text. `experiment` column has values 'Exp1', 'Exp2', 'Exp3' — we encode these as 0, 1, 2 using LabelEncoder.

In [ ]:
le_exp = LabelEncoder()
df_model['experiment'] = le_exp.fit_transform(df_model['experiment'])
print('Experiment encoding:', dict(zip(le_exp.classes_, le_exp.transform(le_exp.classes_))))

# Encode target label
le_label = LabelEncoder()
df_model['label_encoded'] = le_label.fit_transform(df_model[TARGET_COL])
print('Label encoding:', dict(zip(le_label.classes_, le_label.transform(le_label.classes_))))
print()
print('Final feature dtypes:')
print(df_model[FEATURE_COLS].dtypes)

## Step 6 – Participant-wise Train/Test Split

**Why participant-wise?**
Each participant has ~1,152 rows. If we split randomly, the same person's data appears in both train AND test sets. The model learns that person's individual response pattern — not the general pattern. This is **data leakage** and causes falsely high accuracy.

**Solution:** `GroupShuffleSplit` — ensures all rows of one participant go to EITHER train OR test, never both.

- Train: 80% of participants
- Test: 20% of participants

In [ ]:
X = df_model[FEATURE_COLS].values
y = df_model['label_encoded'].values
groups = df_model[GROUP_COL].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_participants = set(groups[train_idx])
test_participants  = set(groups[test_idx])

print(f'Train: {len(X_train):,} rows | {len(train_participants)} participants')
print(f'Test : {len(X_test):,} rows  | {len(test_participants)} participants')
print(f'Overlap (must be 0): {len(train_participants & test_participants)}')
print()
print('Train label distribution:')
for code, name in zip(le_label.transform(le_label.classes_), le_label.classes_):
    count = (y_train == code).sum()
    print(f'  {name}: {count:,}')
print('Test label distribution:')
for code, name in zip(le_label.transform(le_label.classes_), le_label.classes_):
    count = (y_test == code).sum()
    print(f'  {name}: {count:,}')

## Step 7 – Save preprocessed data

Save to Drive so Day 3 (model training) can load directly without repeating all steps.

In [ ]:
import numpy as np

np.save(os.path.join(OUT_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(OUT_DIR, 'X_test.npy'),  X_test)
np.save(os.path.join(OUT_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(OUT_DIR, 'y_test.npy'),  y_test)

# Save label names for later reporting
import json
meta = {
    'feature_cols': FEATURE_COLS,
    'label_names': list(le_label.classes_),
    'experiment_encoding': dict(zip(le_exp.classes_, [int(x) for x in le_exp.transform(le_exp.classes_)])),
    'tercile_low_ms': float(t33),
    'tercile_high_ms': float(t67),
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'train_participants': int(len(train_participants)),
    'test_participants': int(len(test_participants))
}
with open(os.path.join(OUT_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to:', OUT_DIR)
print('Files:', os.listdir(OUT_DIR))
print()
print('meta.json:')
print(json.dumps(meta, indent=2))

## Summary – Day 2

| Step | What we did | Result |
|------|------------|--------|
| Outlier removal | Removed RT < 100ms and RT > Q3+3×IQR | Cleaner distribution |
| Re-labelling | Recalculated terciles on clean data | New Low/Med/High boundaries |
| Feature selection | Kept 7 features, dropped 15+ irrelevant columns | Clean feature matrix |
| Encoding | LabelEncoded `experiment` and `difficulty` | All values numeric |
| Participant split | GroupShuffleSplit 80/20 by participant | 0 overlap between train/test |
| Save | X_train, X_test, y_train, y_test saved as .npy | Ready for Day 3 |

**→ Next: Day 3 – Train Logistic Regression baseline classifier**